In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil

# Yollar
drive_project_path = '/content/drive/MyDrive/DiffuVQA'
local_clone_path = '/content/DiffuVQA'
data_set = "SLAKE"


# ADIM 1: Colab'daki geçici klasörü temizle (Taze indirme için şart)
if os.path.exists(local_clone_path):
    print(f"Eski geçici klasör siliniyor: {local_clone_path}")
    shutil.rmtree(local_clone_path)
# ADIM 2: GitHub'dan en güncel hali çek
print("GitHub'dan taze kopya çekiliyor...")
repo_url = "https://github.com/panzerofthelake03/DiffuVQA.git"
branch = "Bert"

!git clone -b {branch} --single-branch {repo_url} {local_clone_path}
# ADIM 3: Drive'ı GÜNCELLE (En kritik kısım)
# dirs_exist_ok=True sayesinde klasör olsa bile içine girip dosyaları yeniler.
print(f"Drive'daki dosyalar güncelleniyor: {drive_project_path}")

if not os.path.exists(drive_project_path):
    os.makedirs(drive_project_path)

shutil.copytree(local_clone_path, drive_project_path, dirs_exist_ok=True)

print("\n✅ İŞLEM TAMAM: Arkadaşının attığı yeni dosyalar Drive'a geçti!")

Eski geçici klasör siliniyor: /content/DiffuVQA
GitHub'dan taze kopya çekiliyor...
Cloning into '/content/DiffuVQA'...
remote: Enumerating objects: 789, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 789 (delta 203), reused 186 (delta 185), pack-reused 542 (from 2)
Receiving objects: 100% (789/789), 106.99 MiB | 18.56 MiB/s, done.
Resolving deltas: 100% (363/363), done.
Updating files: 100% (422/422), done.
Drive'daki dosyalar güncelleniyor: /content/drive/MyDrive/DiffuVQA

✅ İŞLEM TAMAM: Arkadaşının attığı yeni dosyalar Drive'a geçti!


In [ ]:
import os
import shutil
from tqdm.auto import tqdm


# Assuming the SLAKE image data is located at a specific path on Google Drive.
# IMPORTANT: PLEASE UPDATE 'source_slake_image_path' WITH THE ACTUAL PATH TO YOUR SLAKE IMAGES.
# Example: '/content/drive/MyDrive/SLAKE_DATA/imgs' or '/content/drive/MyDrive/datasets/slake/imgs'
source_slake_image_path = f"{drive_project_path}/datasets/{data_set}/imgs" # <--- UPDATE THIS PATH!

# 'drive_project_path' and 'data_set' variables are already defined in previous cells.

if data_set.lower() == 'slake':
    destination_slake_image_path = os.path.join(local_clone_path, 'datasets')

    print(f"Checking if source SLAKE image path exists: {source_slake_image_path}")
    if os.path.exists(source_slake_image_path):
        os.makedirs(destination_slake_image_path, exist_ok=True)
        print(f"Copying SLAKE images from '{source_slake_image_path}' to '{destination_slake_image_path}'...")

        # Build a file list first so tqdm can show total progress.
        files_to_copy = []
        for root, _, files in os.walk(source_slake_image_path):
            for file_name in files:
                src_file = os.path.join(root, file_name)
                rel_path = os.path.relpath(src_file, source_slake_image_path)
                dst_file = os.path.join(destination_slake_image_path, rel_path)
                files_to_copy.append((src_file, dst_file))

        for src_file, dst_file in tqdm(files_to_copy, desc="Copying SLAKE files", unit="file"):
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.copy2(src_file, dst_file)

        print("✅ SLAKE images copied successfully!")
    else:
        print(f"❌ Error: Source SLAKE image path not found: '{source_slake_image_path}'. Please update the 'source_slake_image_path' variable in this cell.")
else:
    print(f"Dataset is '{data_set}', no SLAKE-specific image copying needed.")


Checking if source SLAKE image path exists: /content/drive/MyDrive/DiffuVQA/datasets/SLAKE/imgs
Copying SLAKE images from '/content/drive/MyDrive/DiffuVQA/datasets/SLAKE/imgs' to '/content/DiffuVQA/datasets'...


In [21]:
import os

# Projenin Drive'daki yolu
drive_project_path = '/content/drive/MyDrive/DiffuVQA'

# 1. Klasör var mı kontrol et
if os.path.exists(drive_project_path):
    # 2. Çalışma dizinini oraya değiştir
    os.chdir(drive_project_path)
    print(f"📂 Çalışma dizini değiştirildi: {os.getcwd()}")

    # 3. Git Pull komutunu çalıştır
    print("⬇️ GitHub'dan güncellemeler çekiliyor (git pull)...")
    !git pull

    # 4. Son durumu göster
    print("\n✅ Güncelleme tamamlandı. İşte son 3 değişiklik:")
    !git log -3 --oneline
else:
    print("❌ Hata: Belirtilen yolda proje klasörü bulunamadı. Önce clone yapmalısın.")

📂 Çalışma dizini değiştirildi: /content/drive/MyDrive/DiffuVQA
⬇️ GitHub'dan güncellemeler çekiliyor (git pull)...
Already up to date.

✅ Güncelleme tamamlandı. İşte son 3 değişiklik:
2880857 (HEAD -> Bert, origin/Bert) Update on vqa_dataset.py
d9d4d17 fixes on Shared folder
a5ec177 Basic Util changed and moved to Shared folder.


In [22]:
import os

print("Current directory:", os.getcwd())

req_file = 'requirements.txt'
colab_req_file = 'requirements_colab.txt'

if os.path.exists(req_file):
    print(f"Processing {req_file} to fix conflicts...")

    with open(req_file, 'r') as f:
        lines = f.readlines()

    # --- ZİNCİRİ KIRAN FİLTRELEME ---
    # 1. torch & torchvision: Colab'ın kendi sürümleri kalsın.
    # 2. tokenizers: Eski sürüm hata veriyor, siliyoruz.
    # 3. transformers: Dosyadaki istek eski sürümü (4.22.2) çekiyor ve bu da eski tokenizers'ı zorluyor.
    #    Biz zaten elle güncel transformers kurduk, o yüzden bunu da listeden siliyoruz.
    cleaned_lines = []
    for line in lines:
        if any(bad_lib in line for bad_lib in ['torch', 'torchvision', 'tokenizers', 'transformers']):
            print(f"Skipping conflicting line: {line.strip()}")
        else:
            cleaned_lines.append(line)

    with open(colab_req_file, 'w') as f:
        f.writelines(cleaned_lines)

    print(f"\nInstalling from clean file: {colab_req_file}...")
    !pip install -r {colab_req_file}

    print("\n✅ Kurulum Başarıyla Tamamlandı!")
else:
    print(f"❌ {req_file} not found.")

Current directory: /content/drive/MyDrive/DiffuVQA
Processing requirements.txt to fix conflicts...
Skipping conflicting line: # NOTE: For CUDA-enabled PyTorch, install the correct wheel from https://pytorch.org/ (select your CUDA)
Skipping conflicting line: # pip install "torch==1.13.1+cu117" -f https://download.pytorch.org/whl/torch_stable.html
Skipping conflicting line: torch>=1.13.1
Skipping conflicting line: torchvision>=0.14.1
Skipping conflicting line: transformers>=4.22.2
Skipping conflicting line: tokenizers
Skipping conflicting line: torchmetrics
Skipping conflicting line: sentence-transformers>=2.0.0  # For semantic similarity metrics
Skipping conflicting line: torch==1.13.1+cu117
Skipping conflicting line: torchmetrics
Skipping conflicting line: transformers==4.22.2

Installing from clean file: requirements_colab.txt...

✅ Kurulum Başarıyla Tamamlandı!


In [23]:

import sys
sys.path.append('.')

# quick smoke test: import new modules
try:
    from diffuvqa.language_encoders.biogpt_model import BioGPTWrapper
    from basic_utils import load_defaults_config
    print('imports OK')
    cfg = load_defaults_config()
    print('loaded defaults config:', cfg.get('config_name'))
    # instantiate AutoConfig to validate config name
    from transformers import AutoConfig
    name = cfg.get('config_name')
    print('creating AutoConfig for', name)
    AutoConfig.from_pretrained(name)
    print('AutoConfig OK')
except Exception as e:
    print('ERROR', repr(e))

imports OK
loaded defaults config: bert-base-uncased
creating AutoConfig for bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

AutoConfig OK


In [24]:
import os

# Drive üzerindeki proje kök dizini
drive_root = '/content/drive/MyDrive/DiffuVQA'
batch_size = 2
lr_rate = 0.0001
learning_steps = 2000
data_set = "slake"

# Checkpointlerin kaydedileceği TAM YOL
checkpoint_dir = os.path.join(drive_root, f'checkpoints/{data_set}/{branch}/btch{batch_size}/lr{lr_rate}')
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Eğitim başlıyor... Modeller şuraya kaydedilecek: {checkpoint_dir}")

# Çalışma dizinini garantiye alalım
os.chdir(drive_root)

!python train.py \
  --lr {lr_rate} \
  --batch_size {batch_size} \
  --learning_steps {learning_steps} \
  --save_interval 200 \
  --log_interval 50 \
  --data_dir datasets \
  --image_dir datasets/slake/imgs \
  --dataset {data_set} \
  --checkpoint_path {checkpoint_dir}

Eğitim başlıyor... Modeller şuraya kaydedilecek: /content/drive/MyDrive/DiffuVQA/checkpoints/slake/Bert/btch2/lr0.0001
Logging to /tmp/openai-2026-03-05-10-48-07-562326
### Creating data loader...
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 143kB/s]
vocab.txt: 232kB [00:00, 571kB/s]
tokenizer.json: 466kB [00:00, 1.44MB/s]
save tokenizer to /content/drive/MyDrive/DiffuVQA/checkpoints/slake/Bert/btch2/lr0.0001
initializing the random embeddings Embedding(30522, 768)
##############################
Loading text data...
##############################
Loading dataset slake from datasets...
### Loading from the TRAIN set...
### Data samples...
questions: ['What modality is used to take this image?', 'Which part of the body does this image belong to?'] answers: ['MRI', 'Abdomen'] images: ['imgs/xmlab1/source.jpg', 'imgs/xmlab1/source.jpg']
### Saved sample preview to datasets/slake_train_first3_preview.png
Figure(1500x500)
RAM used: 1014.44 MB
Dataset size: 4919
Tokenized 512 / 4919 --

In [16]:
model_path = f"{checkpoint_dir}/ema_0.9999_{learning_steps:06d}.pt"

!python sample_vqa_GPU.py --model_path {model_path} --batch_size 4 --top_p -1 --out_dir "/content/drive/MyDrive/DiffuVQA/samples/slake/main" --seed 125   --step 5 --dataset davekevin

Logging to /tmp/openai-2026-03-05-10-46-27-433181
DEBUG: Parsed model_path = /content/drive/MyDrive/DiffuVQA/checkpoints/slake/BioMedLM/btch2/lr0.0001/ema_0.9999_002000.pt
### Loading model from /content/drive/MyDrive/DiffuVQA/checkpoints/slake/BioMedLM/btch2/lr0.0001/ema_0.9999_002000.pt
config_path: /content/drive/MyDrive/DiffuVQA/checkpoints/slake/BioMedLM/btch2/lr0.0001/training_args.json
### Updated args: Namespace(model_path='/content/drive/MyDrive/DiffuVQA/checkpoints/slake/BioMedLM/btch2/lr0.0001/ema_0.9999_002000.pt', step=5, out_dir='/content/drive/MyDrive/DiffuVQA/samples/slake/main', top_p=-1, lr=0.0001, batch_size=4, learning_steps=2000, microbatch=64, log_interval=50, save_interval=200, eval_interval=500, valid=True, ema_rate='0.9999', resume_checkpoint='none', schedule_sampler='lossaware', diffusion_steps=2500, noise_schedule='sqrt', timestep_respacing=[2500], vocab='roberta', use_plm_init='roberta', vocab_size=50265, config_name='roberta-large', notes='folder-notes', da

In [17]:
!pip install torchmetrics

In [18]:
!python eval_DiffuVQA.py --folder "/content/drive/MyDrive/DiffuVQA/samples/slake/BioMedLM" --filename "ema_0.9999_002000.pt.seed125_step0_samplestep5_bsize4.jsonl"

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Enhanced metrics loaded successfully!
Excel export module loaded successfully!
save tokenizer to diffuvqa/config
/content/drive/MyDrive/DiffuVQA/samples/slake/BioMedLM
Traceback (most recent call last):
  File "/content/drive/MyDrive/DiffuVQA/eval_DiffuVQA.py", line 218, in <module>
    with open(files[0], 'r') as f:
         ^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DiffuVQA/samples/slake/BioMedLM/ema_0.9999_002000.pt.seed125_step0_samplestep5_bsize4.jsonl'
